In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.special import gammaln
from scipy.optimize import minimize
from scipy.stats import chi2, poisson
import warnings
import os # Import the os module
warnings.filterwarnings('ignore')

# ============================================================
# 1. DATA
# ============================================================

# Frekuensi Klaim per bulan (kolom "Frekuensi Klaim", Waktu = serial Excel)
# Serial Excel -> tanggal:
# 44562 = Jan 2022, dst.
# Nilai 373888 pada kedatangan klaim adalah anomali (kemungkinan typo), diabaikan.

waktu_serial = [
    44562, 44593, 44621, 44652, 44682, 44713, 44743, 44774, 44805,
    44835, 44866, 44896, 44927, 44958, 44986, 45017, 45047,
    45078, 45108, 45139, 45170, 45200, 45231, 45261, 45292, 45323,
    45352, 45383, 45413, 45444, 45474, 45505, 45536, 45566, 45597,
    45627
]

frekuensi = [
    0, 0, 0, 0, 0, 3, 1, 5, 0, 0,
    13, 0, 0, 3, 2, 0, 0, 0, 0, 0,
    6, 0, 0, 4, 0, 0, 0, 0, 0, 0,
    3, 0, 0, 0, 0, 0
]

# Konversi serial Excel ke datetime (epoch Excel = 1899-12-30)
tanggal = pd.to_datetime('1899-12-30') + pd.to_timedelta(waktu_serial, unit='D')

df = pd.DataFrame({'Tanggal': tanggal, 'Frekuensi': frekuensi})
n = len(frekuensi)
data = np.array(frekuensi)

# ============================================================
# 2. STATISTIKA DESKRIPTIF
# ============================================================

print("=" * 60)
print("         STATISTIKA DESKRIPTIF FREKUENSI KLAIM")
print("=" * 60)
print(f"  Jumlah periode pengamatan (n)  : {n}")
print(f"  Total klaim                    : {data.sum()}")
print(f"  Mean (rata-rata)               : {data.mean():.4f}")
print(f"  Median                         : {np.median(data):.4f}")
print(f"  Modus                          : {pd.Series(data).mode().values}")
print(f"  Variansi                       : {data.var(ddof=1):.4f}")
print(f"  Standar Deviasi                : {data.std(ddof=1):.4f}")
print(f"  Minimum                        : {data.min()}")
print(f"  Maksimum                       : {data.max()}")
print(f"  Range                          : {data.max() - data.min()}")
print(f"  Skewness                       : {pd.Series(data).skew():.4f}")
print(f"  Kurtosis                       : {pd.Series(data).kurt():.4f}")
print(f"  Rasio Variansi/Mean (Indeks D) : {data.var(ddof=1)/data.mean():.4f}")
print()
print("  Catatan: Indeks Dispersi > 1 mengindikasikan overdispersi,")
print("  yang mendukung penggunaan distribusi Negative Binomial atau ZIP.")
print()

# Tabel distribusi frekuensi observasi
nilai_unik = np.arange(0, data.max() + 1)
freq_obs = np.array([(data == k).sum() for k in nilai_unik])
print("  Tabel Distribusi Frekuensi Observasi:")
print(f"  {'k':>4}  {'f(k)':>6}  {'Proporsi':>10}")
print("  " + "-" * 26)
for k, f in zip(nilai_unik, freq_obs):
    print(f"  {k:>4}  {f:>6}  {f/n:>10.4f}")
print()

# ============================================================
# 3. GRAFIK
# ============================================================

# Create the output directory if it doesn't exist
output_dir = '/mnt/user-data/outputs'
os.makedirs(output_dir, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Time Series ---
ax1 = axes[0]
ax1.plot(df['Tanggal'], df['Frekuensi'], marker='o', linewidth=1.5,
         color='steelblue', markersize=4, label='Frekuensi Klaim')
ax1.fill_between(df['Tanggal'], df['Frekuensi'], alpha=0.2, color='steelblue')
ax1.axhline(data.mean(), color='red', linestyle='--', linewidth=1.2,
            label=f'Rata-rata = {data.mean():.2f}')
ax1.set_title('Time Series Frekuensi Klaim per Bulan', fontsize=12, fontweight='bold')
ax1.set_xlabel('Periode (Bulan-Tahun)')
ax1.set_ylabel('Jumlah Klaim')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Histogram ---
ax2 = axes[1]
bins = np.arange(-0.5, data.max() + 1.5, 1)
ax2.hist(data, bins=bins, color='steelblue', edgecolor='white',
         linewidth=0.8, rwidth=0.85)
ax2.set_title('Histogram Frekuensi Klaim', fontsize=12, fontweight='bold')
ax2.set_xlabel('Jumlah Klaim (k)')
ax2.set_ylabel('Frekuensi Kemunculan')
ax2.set_xticks(range(0, data.max() + 1))
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'grafik_klaim.png'), dpi=150, bbox_inches='tight')
plt.close()
print("  [Grafik tersimpan: grafik_klaim.png]")
print()

# ============================================================
# 4. ESTIMASI PARAMETER MLE
# ============================================================

# ----------------------------------------------------------
# 4a. Distribusi POISSON
# ----------------------------------------------------------
# PDF: P(N=k) = e^(-lambda) * lambda^k / k!
# MLE lambda = x_bar

lambda_mle = data.mean()

print("=" * 60)
print("         ESTIMASI PARAMETER MLE")
print("=" * 60)
print()
print("  [1] Distribusi POISSON")
print(f"      lambda (MLE) = {lambda_mle:.6f}")

def loglik_poisson(params):
    lam = params[0]
    if lam <= 0:
        return np.inf
    ll = np.sum(data * np.log(lam) - lam - gammaln(data + 1))
    return -ll

ll_poisson = -loglik_poisson([lambda_mle])
print(f"      Log-Likelihood = {ll_poisson:.6f}")
k_poisson = 1  # jumlah parameter
print()

# ----------------------------------------------------------
# 4b. Distribusi NEGATIVE BINOMIAL — sesuai persamaan (2.18)
# P(N=k) = C(k+r-1, k) * (beta/(beta+t))^r * (t/(beta+t))^k
# E[N] = r*t/beta ; Var[N] = r*(beta+t)/beta^2   (t=1)
# ----------------------------------------------------------

def negloglik_nb(params):
    r, beta = params
    t = 1.0
    if r <= 0 or beta <= 0:
        return np.inf
    p = beta / (beta + t)   # dipangkatkan r  (sesuai 2.18)
    q = t / (beta + t)      # dipangkatkan k  (sesuai 2.18)
    ll = 0.0
    for k in data:
        ll += (gammaln(k + r) - gammaln(r) - gammaln(k + 1)
               + r * np.log(p) + k * np.log(q))
    return -ll

# Initial guess (method of moments) — diturunkan ulang untuk (2.18):
# mean = r/beta, var = r(beta+1)/beta^2 (t=1)
# => var - mean = mean/beta  =>  beta = mean/(var-mean)
xbar = data.mean()
s2 = data.var(ddof=1)
if s2 > xbar:
    beta_init = xbar / (s2 - xbar)
    r_init = xbar * beta_init
else:
    beta_init = 1.0
    r_init = 1.0

result_nb = minimize(negloglik_nb, x0=[r_init, beta_init],
                     method='Nelder-Mead',
                     options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 10000})

r_mle, beta_mle = result_nb.x
ll_nb = -result_nb.fun
k_nb = 2

print("  [2] Distribusi NEGATIVE BINOMIAL (sesuai persamaan 2.18)")
print(f"      r    (MLE) = {r_mle:.6f}")
print(f"      beta (MLE) = {beta_mle:.6f}")
print(f"      Log-Likelihood = {ll_nb:.6f}")
print(f"      Verifikasi E[N] = r/beta = {r_mle/beta_mle:.6f} | Mean sampel = {xbar:.6f}")
print()

# ----------------------------------------------------------
# 4c. Zero-Inflated Poisson (ZIP)
# P(N=0) = pi + (1-pi)*e^(-lambda)
# P(N=k) = (1-pi)*e^(-lambda)*lambda^k/k!, k >= 1
# ----------------------------------------------------------

def negloglik_zip(params):
    pi, lam = params
    if pi < 0 or pi >= 1 or lam <= 0:
        return np.inf
    ll = 0.0
    for k in data:
        if k == 0:
            p0 = pi + (1 - pi) * np.exp(-lam)
            ll += np.log(p0 + 1e-300)
        else:
            pk = (1 - pi) * np.exp(-lam) * lam**k / np.exp(gammaln(k + 1))
            ll += np.log(pk + 1e-300)
    return -ll

# Initial guess
prop_zeros = (data == 0).mean()
lam_init = data[data > 0].mean() if (data > 0).any() else 1.0
pi_init = max(prop_zeros - np.exp(-lam_init), 0.01)
pi_init = min(pi_init, 0.95)

result_zip = minimize(negloglik_zip, x0=[pi_init, lam_init],
                      method='Nelder-Mead',
                      options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 10000})

pi_mle, lambda_zip_mle = result_zip.x
ll_zip = -result_zip.fun
k_zip = 2  # jumlah parameter

print("  [3] Distribusi ZERO-INFLATED POISSON (ZIP)")
print(f"      pi     (MLE) = {pi_mle:.6f}")
print(f"      lambda (MLE) = {lambda_zip_mle:.6f}")
print(f"      Log-Likelihood = {ll_zip:.6f}")
print()

# ============================================================
# 5. PROBABILITAS TEORITIS & FREKUENSI EKSPEKTASI
# ============================================================

def prob_poisson(k, lam):
    return np.exp(-lam) * lam**k / np.exp(gammaln(k + 1))

def prob_nb(k, r, beta, t=1.0):
    p = beta / (beta + t)   # dipangkatkan r
    q = t / (beta + t)      # dipangkatkan k
    log_p = (gammaln(k + r) - gammaln(r) - gammaln(k + 1)
             + r * np.log(p) + k * np.log(q))
    return np.exp(log_p)

def prob_zip(k, pi, lam):
    if k == 0:
        return pi + (1 - pi) * np.exp(-lam)
    else:
        return (1 - pi) * np.exp(-lam) * lam**k / np.exp(gammaln(k + 1))

# Hitung probabilitas untuk setiap nilai k
max_k = int(data.max())
k_vals = np.arange(0, max_k + 1)

# Gabungkan ekor jika frekuensi ekspektasi < 5 (aturan Chi-Square)
def hitung_ekspektasi(prob_func, **kwargs):
    probs = np.array([prob_func(k, **kwargs) for k in k_vals])
    # Pastikan total prob = 1 (normalisasi ekor)
    prob_ekor = 1 - probs.sum()
    return probs * n, prob_ekor * n

# ============================================================
# 6. CHI-SQUARE GOODNESS OF FIT
# ============================================================

def chi_square_test(prob_func, params_dict, k_params, label):
    """
    Melakukan uji Chi-Square dengan penggabungan sel yang E_i < 5.
    """
    probs = np.array([prob_func(k, **params_dict) for k in k_vals])
    # Prob untuk k > max_k (ekor kanan)
    prob_tail = 1.0 - probs.sum()

    # Frekuensi observasi & ekspektasi
    obs = np.array([(data == k).sum() for k in k_vals])
    obs_all = np.append(obs, n - obs.sum())       # ekor observasi = sisa
    exp_all = np.append(probs * n, prob_tail * n) # ekspektasi termasuk ekor
    k_all   = list(k_vals) + [f'>{max_k}']

    # Gabungkan sel dengan E < 5 (dari kanan)
    obs_merged = list(obs_all)
    exp_merged = list(exp_all)
    labels_merged = list(k_all)

    i = len(exp_merged) - 1
    while i > 0 and exp_merged[i] < 5:
        exp_merged[i-1] += exp_merged[i]
        obs_merged[i-1] += obs_merged[i]
        exp_merged.pop(i)
        obs_merged.pop(i)
        labels_merged.pop(i)
        i -= 1

    obs_m = np.array(obs_merged)
    exp_m = np.array(exp_merged)

    # Hapus sel dengan exp = 0 untuk menghindari division by zero
    mask = exp_m > 0
    obs_m = obs_m[mask]
    exp_m = exp_m[mask]

    chi2_stat = np.sum((obs_m - exp_m)**2 / exp_m)
    df_chi = len(obs_m) - 1 - k_params  # derajat kebebasan
    df_chi = max(df_chi, 1)
    p_val = 1 - chi2.cdf(chi2_stat, df_chi)

    print(f"  Uji Chi-Square: {label}")
    print(f"  {'Sel k':<12} {'O (Obs)':>8} {'E (Eksp)':>10} {'(O-E)^2/E':>12}")
    print("  " + "-" * 46)
    for lbl, o, e in zip(labels_merged, obs_m, exp_m):
        kontrib = (o - e)**2 / e
        print(f"  {str(lbl):<12} {o:>8.0f} {e:>10.4f} {kontrib:>12.4f}")
    print("  " + "-" * 46)
    print(f"  Chi-Square Statistik = {chi2_stat:.4f}")
    print(f"  Derajat Kebebasan    = {df_chi}")
    print(f"  P-value              = {p_val:.4f}")
    tolak = "TOLAK H0 (distribusi tidak cocok)" if p_val < 0.05 else "GAGAL TOLAK H0 (distribusi cocok)"
    print(f"  Kesimpulan (α=0.05)  : {tolak}")
    print()

    return chi2_stat, df_chi, p_val

print("=" * 60)
print("         UJI CHI-SQUARE GOODNESS OF FIT")
print("=" * 60)
print()

chi2_pois, df_pois, p_pois = chi_square_test(
    prob_poisson, {'lam': lambda_mle}, k_poisson, "Distribusi Poisson")

chi2_nb, df_nb, p_nb = chi_square_test(
    prob_nb, {'r': r_mle, 'beta': beta_mle}, k_nb, "Distribusi Negative Binomial")

chi2_zip, df_zip, p_zip = chi_square_test(
    prob_zip, {'pi': pi_mle, 'lam': lambda_zip_mle}, k_zip, "Distribusi ZIP")

# ============================================================
# 7. AIC & BIC
# ============================================================

def hitung_aic(ll, k):
    return 2 * k - 2 * ll

def hitung_bic(ll, k, n):
    return k * np.log(n) - 2 * ll

aic_pois = hitung_aic(ll_poisson, k_poisson)
aic_nb   = hitung_aic(ll_nb, k_nb)
aic_zip  = hitung_aic(ll_zip, k_zip)

bic_pois = hitung_bic(ll_poisson, k_poisson, n)
bic_nb   = hitung_bic(ll_nb, k_nb, n)
bic_zip  = hitung_bic(ll_zip, k_zip, n)

print("=" * 60)
print("         AIC DAN BIC")
print("=" * 60)
print()
print(f"  {'Distribusi':<22} {'k':>4}  {'Log-L':>10}  {'AIC':>10}  {'BIC':>10}")
print("  " + "-" * 62)
print(f"  {'Poisson':<22} {k_poisson:>4}  {ll_poisson:>10.4f}  {aic_pois:>10.4f}  {bic_pois:>10.4f}")
print(f"  {'Negative Binomial':<22} {k_nb:>4}  {ll_nb:>10.4f}  {aic_nb:>10.4f}  {bic_nb:>10.4f}")
print(f"  {'ZIP':<22} {k_zip:>4}  {ll_zip:>10.4f}  {aic_zip:>10.4f}  {bic_zip:>10.4f}")
print()

# Pilih model terbaik berdasarkan AIC terkecil
aic_dict = {'Poisson': aic_pois, 'Negative Binomial': aic_nb, 'ZIP': aic_zip}
bic_dict = {'Poisson': bic_pois, 'Negative Binomial': bic_nb, 'ZIP': bic_zip}
best_aic = min(aic_dict, key=aic_dict.get)
best_bic = min(bic_dict, key=bic_dict.get)
print(f"  Model terbaik (AIC terkecil) : {best_aic}  (AIC = {aic_dict[best_aic]:.4f})")
print(f"  Model terbaik (BIC terkecil) : {best_bic}  (BIC = {bic_dict[best_bic]:.4f})")
print()

# ============================================================
# 8. GRAFIK PERBANDINGAN DISTRIBUSI
# ============================================================

k_plot = np.arange(0, max_k + 1)
obs_prop = np.array([(data == k).sum() / n for k in k_plot])
pois_prop = np.array([prob_poisson(k, lambda_mle) for k in k_plot])
nb_prop   = np.array([prob_nb(k, r_mle, beta_mle) for k in k_plot])
zip_prop  = np.array([prob_zip(k, pi_mle, lambda_zip_mle) for k in k_plot])

fig2, ax = plt.subplots(figsize=(10, 5))
w = 0.2
ax.bar(k_plot - 1.5*w, obs_prop, w, label='Observasi', color='steelblue')
ax.bar(k_plot - 0.5*w, pois_prop, w, label='Poisson', color='tomato', alpha=0.8)
ax.bar(k_plot + 0.5*w, nb_prop,   w, label='Neg. Binomial', color='seagreen', alpha=0.8)
ax.bar(k_plot + 1.5*w, zip_prop,  w, label='ZIP', color='darkorange', alpha=0.8)
ax.set_xlabel('k (jumlah klaim)')
ax.set_ylabel('Proporsi')
ax.set_title('Perbandingan Distribusi Teoritis vs Observasi', fontsize=12, fontweight='bold')
ax.set_xticks(k_plot)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'perbandingan_distribusi.png'), dpi=150, bbox_inches='tight')
plt.close()
print("  [Grafik tersimpan: perbandingan_distribusi.png]")

print()
print("=" * 60)
print("  SELESAI")
print("=" * 60)


         STATISTIKA DESKRIPTIF FREKUENSI KLAIM
  Jumlah periode pengamatan (n)  : 36
  Total klaim                    : 40
  Mean (rata-rata)               : 1.1111
  Median                         : 0.0000
  Modus                          : [0]
  Variansi                       : 6.6730
  Standar Deviasi                : 2.5832
  Minimum                        : 0
  Maksimum                       : 13
  Range                          : 13
  Skewness                       : 3.2715
  Kurtosis                       : 12.6662
  Rasio Variansi/Mean (Indeks D) : 6.0057

  Catatan: Indeks Dispersi > 1 mengindikasikan overdispersi,
  yang mendukung penggunaan distribusi Negative Binomial atau ZIP.

  Tabel Distribusi Frekuensi Observasi:
     k    f(k)    Proporsi
  --------------------------
     0      27      0.7500
     1       1      0.0278
     2       1      0.0278
     3       3      0.0833
     4       1      0.0278
     5       1      0.0278
     6       1      0.0278
     7       0 